# Multi-Agent Sales Workflow with Agentic Governance

This notebook demonstrates real-time node-level evaluation using watsonx.governance AgenticEvaluator.

## Key Features
- Node-level evaluation with decorators
- Real-time metrics during workflow execution
- Graph visualization
- Comprehensive metrics DataFrame with costs, latency, and quality scores

## Setup

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import uuid
from typing import TypedDict

load_dotenv()

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Initialize watsonx.governance

In [ ]:
from ibm_watsonx_gov.evaluators.agentic_evaluator import AgenticEvaluator
from ibm_watsonx_gov.config import AgenticAIConfiguration
from ibm_watsonx_gov.metrics import FaithfulnessMetric, ContextRelevanceMetric
from ibm_watsonx_gov.entities.foundation_model import WxAIFoundationModel
from ibm_watsonx_gov.entities.llm_judge import LLMJudge

PROJECT_ID = os.getenv("WATSONX_PROJECT_ID")
REGION = "us-south"

# Initialize LLM Judge
llm_judge = LLMJudge(
    model=WxAIFoundationModel(
        model_id="meta-llama/llama-3-3-70b-instruct",
        project_id=PROJECT_ID,
        region=REGION
    )
)

# Initialize Agentic Evaluator
evaluator = AgenticEvaluator(
    project_id=PROJECT_ID,
    region=REGION
)

print("[SUCCESS] watsonx.governance initialized")

## Initialize LLM

In [ ]:
# Set WATSONX_APIKEY in environment if not already set
if 'WATSONX_APIKEY' not in os.environ:
    os.environ['WATSONX_APIKEY'] = os.getenv('WATSONX_APIKEY', '')

print(f"API Key set: {'Yes' if os.environ.get('WATSONX_APIKEY') else 'No'}")
print(f"Project ID: {PROJECT_ID[:8]}..." if PROJECT_ID else "Project ID not set")

In [ ]:
from langchain_ibm import ChatWatsonx

# Initialize ChatWatsonx (uses WATSONX_APIKEY from environment)
parameters = {
    "temperature": 0.3,
    "max_tokens": 500,
}

chat_model = ChatWatsonx(
    model_id="meta-llama/llama-4-maverick-17b-128e-instruct-fp8",
    url="https://us-south.ml.cloud.ibm.com",
    project_id=PROJECT_ID,
    params=parameters,
)

print("[SUCCESS] LLM initialized")

## Define State

In [ ]:
class SalesAgentState(TypedDict):
    input_text: str  # User query
    contract_context: list[str]  # Context from contracts
    crm_context: list[str]  # Context from CRM
    generated_text: str  # Final output
    message_id: str  # Unique tracking ID
    record_id: str  # Record ID for governance

## Define Workflow Nodes with Governance Decorators

In [ ]:
from langgraph.config import RunnableConfig
from langchain_core.prompts import ChatPromptTemplate

# Node 1: Retrieve Contract Context
retrieve_contracts_config = {
    "input_fields": ["input_text"],
    "context_fields": ["contract_context"],
    "message_id": ["message_id"]
}

@evaluator.evaluate_retrieval_quality(
    configuration=AgenticAIConfiguration(**retrieve_contracts_config),
    metrics=[ContextRelevanceMetric(llm_judge=llm_judge)],
    compute_real_time=True
)
def retrieve_contracts(state: SalesAgentState, config: RunnableConfig) -> dict:
    """Retrieve relevant contract information"""
    # Simplified: In real implementation, this would query vector store
    query = state["input_text"]
    
    # Mock contract context
    context = [
        f"Contract for Confluent: watsonx services, expires 2026-05-31, value $250,000",
        f"Previous contract history shows successful renewals in 2023 and 2024"
    ]
    
    return {"contract_context": context}

# Node 2: Retrieve CRM Context
retrieve_crm_config = {
    "input_fields": ["input_text"],
    "context_fields": ["crm_context"],
    "message_id": ["message_id"]
}

@evaluator.evaluate_retrieval_quality(
    configuration=AgenticAIConfiguration(**retrieve_crm_config),
    metrics=[ContextRelevanceMetric(llm_judge=llm_judge)],
    compute_real_time=True
)
def retrieve_crm(state: SalesAgentState, config: RunnableConfig) -> dict:
    """Retrieve relevant CRM information"""
    # Mock CRM context
    context = [
        f"CRM Opportunity: Confluent watsonx Renewal, Stage: Qualified, Owner: Anand Das",
        f"Next Steps: Discussing renewal terms and potential expansion"
    ]
    
    return {"crm_context": context}

# Node 3: Generate Response
generate_config = {
    "input_fields": ["input_text"],
    "context_fields": ["contract_context", "crm_context"],
    "output_fields": ["generated_text"]
}

@evaluator.evaluate_faithfulness(
    configuration=AgenticAIConfiguration(**generate_config),
    metrics=[FaithfulnessMetric(llm_judge=llm_judge)],
    compute_real_time=True
)
def generate_response(state: SalesAgentState, config: RunnableConfig) -> dict:
    """Generate sales recommendation based on context"""
    
    # Combine contexts
    all_context = state.get("contract_context", []) + state.get("crm_context", [])
    context_str = "\n".join(all_context)
    
    prompt = ChatPromptTemplate.from_template(
        "CONTEXT: {context}\n"
        "QUERY: {query}\n"
        "You are a sales assistant. Use the provided context to answer the query. "
        "Only use the provided context. Provide a concise, professional response.\n"
        "ANSWER:"
    )
    
    formatted_prompt = prompt.invoke({"query": state["input_text"], "context": context_str})
    response = chat_model.invoke(formatted_prompt)
    
    return {"generated_text": response.content}

print("[SUCCESS] Workflow nodes defined with governance decorators")

## Build LangGraph Workflow

In [ ]:
from langgraph.graph import START, END, StateGraph

# Build the graph
builder = StateGraph(SalesAgentState)

# Add nodes
builder.add_node("retrieve_contracts", retrieve_contracts)
builder.add_node("retrieve_crm", retrieve_crm)
builder.add_node("generate", generate_response)

# Add edges
builder.add_edge(START, "retrieve_contracts")
builder.add_edge("retrieve_contracts", "retrieve_crm")
builder.add_edge("retrieve_crm", "generate")
builder.add_edge("generate", END)

# Compile
sales_agent = builder.compile()

print("[SUCCESS] LangGraph workflow compiled")

## Visualize Workflow Graph

In [ ]:
from IPython.display import Image, display

display(Image(sales_agent.get_graph(xray=True).draw_mermaid_png()))

## Create Test Queries

In [ ]:
test_queries = pd.DataFrame([
    {
        "input_text": "What contracts do we have with Confluent?",
        "message_id": str(uuid.uuid4()),
        "interaction_id": "test_001"
    },
    {
        "input_text": "What is the status of the Confluent renewal opportunity?",
        "message_id": str(uuid.uuid4()),
        "interaction_id": "test_002"
    },
    {
        "input_text": "When does the Confluent contract expire?",
        "message_id": str(uuid.uuid4()),
        "interaction_id": "test_003"
    }
])

display(test_queries)

## Run Workflow with Governance Evaluation

In [ ]:
batch_results = []
agent_results = []

for _, row in test_queries.iterrows():
    print(f"\n{'='*80}")
    print(f"Processing: {row['input_text']}")
    print(f"{'='*80}")
    
    # Start governance tracking
    evaluator.start_run()
    
    # Run the agent
    result = sales_agent.invoke({
        "input_text": row["input_text"],
        "message_id": row["message_id"],
        "record_id": row["message_id"],
        "interaction_id": row["interaction_id"]
    })
    
    # End governance tracking
    evaluator.end_run()
    
    # Store agent result
    agent_results.append({
        "message_id": row["message_id"],
        "input_text": row["input_text"],
        "generated_text": result["generated_text"]
    })
    
    print(f"\nGenerated Response:\n{result['generated_text']}")
    
    # Get metric results
    eval_result = evaluator.get_result()
    metric_result = eval_result.to_df()
    metric_result["message_id"] = row["message_id"]
    batch_results.append(metric_result)

print(f"\n{'='*80}")
print("[SUCCESS] All queries processed")
print(f"{'='*80}")

## Combine Results into Comprehensive DataFrame

In [ ]:
# Combine all metric results
all_metrics_df = pd.concat(batch_results, ignore_index=True)

# Create agent results DataFrame
agent_df = pd.DataFrame(agent_results)

# Merge agent results with metrics
final_df = agent_df.merge(all_metrics_df, on="message_id", how="left")

print("\n" + "="*80)
print("COMPREHENSIVE RESULTS WITH GOVERNANCE METRICS")
print("="*80)
display(final_df)

# Display summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

# Find faithfulness columns
faithfulness_cols = [col for col in final_df.columns if 'faithfulness' in col.lower()]
if faithfulness_cols:
    for col in faithfulness_cols:
        if pd.api.types.is_numeric_dtype(final_df[col]):
            print(f"\n{col}:")
            print(f"  Average: {final_df[col].mean():.3f}")
            print(f"  Min: {final_df[col].min():.3f}")
            print(f"  Max: {final_df[col].max():.3f}")

# Find context relevance columns
relevance_cols = [col for col in final_df.columns if 'context_relevance' in col.lower() or 'retrieval' in col.lower()]
if relevance_cols:
    for col in relevance_cols:
        if pd.api.types.is_numeric_dtype(final_df[col]):
            print(f"\n{col}:")
            print(f"  Average: {final_df[col].mean():.3f}")
            print(f"  Min: {final_df[col].min():.3f}")
            print(f"  Max: {final_df[col].max():.3f}")

# Find cost and latency columns
cost_cols = [col for col in final_df.columns if 'cost' in col.lower()]
latency_cols = [col for col in final_df.columns if 'latency' in col.lower()]

if cost_cols:
    print(f"\nTotal Cost: ${final_df[cost_cols].sum().sum():.4f}")

if latency_cols:
    for col in latency_cols:
        if pd.api.types.is_numeric_dtype(final_df[col]):
            print(f"\n{col}:")
            print(f"  Average: {final_df[col].mean():.3f}s")
            print(f"  Total: {final_df[col].sum():.3f}s")

## Export Results

In [ ]:
# Export to CSV
output_file = "sales_agent_governance_results.csv"
final_df.to_csv(output_file, index=False)
print(f"[SUCCESS] Results exported to {output_file}")

## Summary

This notebook demonstrates:
1. **Real-time node-level evaluation** using AgenticEvaluator decorators
2. **LangGraph workflow** with contract and CRM retrieval nodes
3. **Graph visualization** showing the workflow structure
4. **Comprehensive metrics** including faithfulness, context relevance, costs, and latency
5. **Batch processing** with governance tracking for multiple queries

### Key Differences from Batch Evaluation:
- Metrics are computed **during** workflow execution, not after
- Each node can be evaluated independently
- Provides detailed insights into each step of the workflow
- Enables conditional routing based on metric scores